In [ ]:
# imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error
import seaborn as sns

In [ ]:
## Config

# select if all individuals should be included or only the subsample with plasma p-tau217 available
subsample = False

# select if the plots should be saved and corresponding save name
save_plots = True 
savename = 'unet_load_res'

In [ ]:
# load the evaluation file (output from examples/evaluate_unet.py) and test set dataframe
df_eval = pd.read_csv('../outputs/eval/evaluation_results.csv')
df_test = pd.read_csv('../datasets/df_test.csv')

if subsample:
    df_test = df_test.dropna(subset='plasma_ptau217')
    
df_eval = df_eval.merge(df_test)

# Specify the hue order and corresponding datasets. Example from article Figure 3.
hue_order = ['BF2','BF1','PREVENT-AD','UCSF','ADNI','A4','BACS',
             'OASIS-3','A04','A05','A08','LLCF','LZAX']
if subsample:
    hue_order = ['BF2','BF1','PREVENT-AD','UCSF','ADNI','A4']

orderdict = {
    'BF2': 0,
    'BF1': 6,
    'PREVENT-AD': 4,
    'UCSF': 5,
    'ADNI': 2,
    'A4': 1,
    'BACS': 3,
    'OASIS-3': 3,
    'A04': 3,
    'A05': 1.5,
    'A08': 0,
    'LLCF': 0.5,
    'LZAX': 3
}

df_eval['study_order'] = df_eval['study'].map(orderdict)
df_eval = df_eval.sort_values(by='study_order')

In [ ]:
# Create scatter plots for Braak regions as in Figure 3 in the article.
figure, axs = plt.subplots(1,3,figsize=(9,2.5),sharey = True)
plt.rc('font', size=12)
plt.subplots_adjust(wspace = 0.1)

names_true = ['I-II_true', 'I-IV_true','V-VI_true']
names_pred = ['I-II_pred', 'I-IV_pred','V-VI_pred']
titles = ['Braak I-II',  'Braak I-IV','Braak V-VI']
props = dict(boxstyle='square', facecolor='white', alpha=0.3)

# select the colors
cmap1 = plt.get_cmap('tab20b')
cmap2 = plt.get_cmap('tab20c')
if subsample:
    colors = [cmap2(0),cmap1(13),cmap1(0),cmap1(10),cmap2(9),cmap1(19)]
    lgd = True
else:
    colors = [cmap2(0),cmap1(13),cmap1(0),cmap1(10),cmap2(9),cmap1(19),
             cmap2(6),cmap1(8),cmap2(17),cmap2(19),cmap2(11),cmap1(15),cmap1(1)]
    lgd = False
    
for ax, name_true, name_pred,marker,title in zip(axs.flat,names_true,names_pred,markers,titles):
    ax.plot([0,5],[0,5],ls='dashed',color='black',lw=1,alpha=0.8)
    sns.scatterplot(df_eval,x=name_true,y=name_pred,hue='study',hue_order=hue_order,
                    ax=ax,legend=lgd,palette=colors,marker='p',s=30)
    ax.axis([0.6,3.5,0.6,3.5])
    ax.set_xticks([1,1.5,2,2.5,3,3.5])
    ax.grid(alpha=0.2)
    ax.set_xlabel('True SUVR')
    ax.set_title(title,weight='bold',size=10)
    # compute correlation
    r = np.corrcoef(df_eval[name_true],df_eval[name_pred])[0,1]
    rsquared = r*r
    #compute mean absolute error
    mae = mean_absolute_error(df_eval[name_true],df_eval[name_pred])
    textstr = 'R = {:.3f} \nR$^2$ = {:.3f} \nMAE = {:.3f}'.format(r,rsquared,mae)
    ax.text(0.05,0.95,textstr,transform=ax.transAxes,fontsize=10,verticalalignment='top',bbox=props)
    if lgd:
        ax.legend(markerscale=1.3,fontsize=7,loc='lower right')

axs[0].set_ylabel('Predicted SUVR')

if save_plots:
    plt.savefig('../outputs/figures/' + savename + '.pdf',bbox_inches='tight')